<a href="https://colab.research.google.com/github/Tobigreat/Crop-Recommender-Systems/blob/main/crop_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#loading libarary
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier


In [ ]:
df = pd.read_csv("/content/croplabel.csv")
df

,N,P,K,temperature,humidity,ph,rainfall,label,state,zone
0,83.923543,59.770692,37.683757,25.653206,79.457024,6.906296,201.370937,rice,Cross River,Humid (High Rainfall)
1,21.902486,60.186877,23.286752,18.683274,20.209139,5.643075,103.881137,kidneybeans,Bauchi,Semi-Arid (Low Rainfall)
2,102.015658,51.759258,19.246464,22.882163,84.003458,7.038660,91.700082,cotton,Kano,Semi-Arid (Low Rainfall)
3,30.590069,120.722812,201.275555,23.167130,90.575176,5.714641,111.293415,apple,Gombe,Semi-Arid (Low Rainfall)
4,5.852636,35.810503,22.139855,24.160242,60.700889,8.870835,41.334045,mothbeans,Jigawa,Semi-Arid (Low Rainfall)
...,...,...,...,...,...,...,...,...,...,...
6995,87.310302,35.986384,24.330482,21.398606,63.227513,6.179786,67.083316,maize,Jigawa,Semi-Arid (Low Rainfall)
6996,19.377504,7.278429,9.395219,27.386320,91.943961,6.978238,100.030231,orange,Kaduna,Semi-Arid (Low Rainfall)
6997,39.036115,8.908054,16.075838,25.261443,91.144193,8.004653,117.443503,orange,Sokoto,Semi-Arid (Low Rainfall)
6998,30.184342,23.787731,11.812968,30.554863,90.899554,7.776129,112.396375,orange,Kano,Semi-Arid (Low Rainfall)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            7000 non-null   float64
 1   P            7000 non-null   float64
 2   K            7000 non-null   float64
 3   temperature  7000 non-null   float64
 4   humidity     7000 non-null   float64
 5   ph           7000 non-null   float64
 6   rainfall     7000 non-null   float64
 7   label        7000 non-null   object 
 8   state        7000 non-null   object 
 9   zone         7000 non-null   object 
dtypes: float64(7), object(3)
memory usage: 547.0+ KB


In [ ]:
df.isnull().sum()

,0
N,0
P,0
K,0
temperature,0
humidity,0
ph,0
rainfall,0
label,0
state,0
zone,0


In [ ]:
df.columns

Index(['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label',
       'state', 'zone'],
      dtype='object')

In [ ]:
df["state"] = df["state"].astype(str).str.strip()
df["zone"] = df["zone"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip()


In [ ]:
target = "label"


In [ ]:
num_cols = ["N","P","K","temperature","humidity","ph","rainfall"]
cat_cols = ["state"]  # you can add "Zone" too


In [ ]:
X = df[num_cols + cat_cols]
y = df[target]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)


In [ ]:
model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    class_weight="balanced"
)


In [ ]:
pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])


In [ ]:
pipe.fit(X_train, y_train)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['state']),
                                                 ('num', 'passthrough',
                                                  ['N', 'P', 'K', 'temperature',
                                                   'humidity', 'ph',
                                                   'rainfall'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=400, random_state=42))])

In [ ]:
y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        64
      banana       1.00      1.00      1.00        64
   blackgram       1.00      1.00      1.00        64
    chickpea       1.00      1.00      1.00        64
     coconut       1.00      1.00      1.00        63
      coffee       1.00      1.00      1.00        63
      cotton       1.00      1.00      1.00        64
      grapes       1.00      1.00      1.00        63
        jute       0.94      1.00      0.97        64
 kidneybeans       1.00      1.00      1.00        63
      lentil       1.00      1.00      1.00        64
       maize       1.00      1.00      1.00        64
       mango       1.00      1.00      1.00        63
   mothbeans       1.00      1.00      1.00        64
    mungbean       1.00      1.00      1.00        64
   muskmelon       1.00      1.00      1.00        64
      orange       1.00      1.00      1.00        64
      papaya       1.00    

In [ ]:
state_crop_prior = (
    df.groupby(["state", "label"])
      .size()
      .div(df.groupby("state").size(), level=0)
      .to_dict()
)


In [ ]:
def recommend_crops(state, N, P, K, temperature, humidity, ph, rainfall, top_k=5, use_state_prior=True):
    # 1) Build a single-row DataFrame with the exact feature columns used in training
    x = pd.DataFrame([{
        "state": str(state).strip(),
        "N": float(N),
        "P": float(P),
        "K": float(K),
        "temperature": float(temperature),
        "humidity": float(humidity),
        "ph": float(ph),
        "rainfall": float(rainfall),
    }])

    # 2) Get probability of each crop class from the trained pipeline
    proba = pipe.predict_proba(x)[0]

    # 3) Get crop class labels in the same order as the probabilities
    crops = pipe.named_steps["model"].classes_

    # 4) Optional: reweight probabilities using how common each crop is in that state (state prior)
    if use_state_prior and x.loc[0, "state"] in df["state"].unique():
        priors = np.array([state_crop_prior.get((x.loc[0, "state"], c), 0.0) for c in crops])

        # Only apply if priors are meaningful (not all zeros)
        if priors.sum() > 0:
            proba = proba * priors
            proba = proba / proba.sum()  # normalize to sum to 1

    # 5) Pick indices of the top-k highest probabilities
    top_idx = np.argsort(proba)[::-1][:top_k]

    # 6) Return (crop, confidence) pairs for top-k crops
    return [(crops[i], float(proba[i])) for i in top_idx]




In [ ]:
def recommend_top_k(context_dict, k=3):
    x = pd.DataFrame([context_dict])
    x = x[num_cols + cat_cols]
    proba = pipe.predict_proba(x)[0]
    classes = pipe.named_steps["model"].classes_
    top_idx = proba.argsort()[::-1][:k]
    return [(classes[i], float(proba[i])) for i in top_idx]

In [ ]:
context = {
    "N": 90, "P": 42, "K": 43,
    "temperature": 26, "humidity": 80,
    "ph": 6.5, "rainfall": 200,
    "state": "Lagos", "Zone": "South West",
    "Season": "Wet"
}
print(recommend_top_k(context, k=3))

[('rice', 0.545), ('jute', 0.445), ('watermelon', 0.005)]


In [ ]:
import joblib

# Save the trained pipeline
joblib.dump(pipe, "crop_recommender_model.pkl")

print("Model saved successfully")

Model saved successfully
